# 1. Intro
Dimensional modeling organizes analytics data into a **star schema**: descriptive **dimensions** around a central **fact table**.

- **Star schema**: simple joins, analyst-friendly, and ideal for BI.
- **Snowflake schema**: more normalized, but usually requires more joins.
- In this project, an LLM helps draft the schema and Python turns that idea into real tables.


In [ ]:
# 2. Load raw orders — show the messy flat file
from pathlib import Path
import sys

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.ingest import load_orders

df = load_orders(project_root / 'data' / 'orders.csv')
df.head(10)


In [ ]:
# 3. Profile the data — show what we send to the LLM
from src.ingest import get_data_profile

profile_text = get_data_profile(df)
print(profile_text)


In [ ]:
# 4. LLM schema suggestion — show the raw JSON response from GPT-4o
# For an offline walkthrough we use the built-in fallback schema.
from src.suggest_schema import get_default_schema
import json

schema = get_default_schema()
print(json.dumps(schema, indent=2))


# 5. Understand the proposal
- `dim_customer` stores reusable customer descriptors.
- `dim_product` stores product hierarchy and price context.
- `dim_date` standardizes time analysis.
- `fact_orders` stores measures at the chosen grain plus foreign keys into the dimensions.


In [ ]:
# 6. Build dimensions — show dim_customer, dim_product, dim_date
from src.build_dims import build_dim_customer, build_dim_product, build_dim_date

dim_customer = build_dim_customer(df)
dim_product = build_dim_product(df)
dim_date = build_dim_date(df)

print('dim_customer')
display(dim_customer.head())
print('dim_product')
display(dim_product.head())
print('dim_date')
display(dim_date.head())


In [ ]:
# 7. Build fact table — show how surrogate keys are joined in
from src.build_facts import build_fact_orders

fact_orders = build_fact_orders(df, dim_customer, dim_product, dim_date)
fact_orders.head()


In [ ]:
# 8. Validate — show the referential integrity check
from src.validate_model import validate_referential_integrity

validation = validate_referential_integrity(fact_orders, dim_customer, dim_product, dim_date)
validation


In [ ]:
# 9. Query the model — sample star schema SQL against DuckDB
import duckdb
from src.build_dims import save_dimensions
from src.build_facts import save_fact_table

output_dir = project_root / 'output'
save_dimensions({'dim_customer': dim_customer, 'dim_product': dim_product, 'dim_date': dim_date}, output_dir)
save_fact_table(fact_orders, output_dir)

con = duckdb.connect()
query = f'''
SELECT
    d.year,
    d.month_name,
    p.product_category,
    SUM(f.total_amount) AS revenue,
    SUM(f.quantity) AS units_sold
FROM read_parquet('{output_dir / 'fact_orders.parquet'}') AS f
JOIN read_parquet('{output_dir / 'dim_date.parquet'}') AS d ON f.date_key = d.date_key
JOIN read_parquet('{output_dir / 'dim_product.parquet'}') AS p ON f.product_key = p.product_key
GROUP BY 1, 2, 3
ORDER BY revenue DESC
LIMIT 10
'''
con.sql(query).df()
